In [ ]:
# ruff: noqa: F401
import json
import os

import pandas as pd
from experiment_utils import *
from sklearn.model_selection import StratifiedShuffleSplit

from k_anonymization.algorithms.full_generalization import *
from k_anonymization.algorithms.local_recoding import *
from k_anonymization.core import Dataset, SampleDataset
from k_anonymization.datasets import HOUSING

In [ ]:
__DATASET = HOUSING
TARGETS = {
    "reg": "price",
    "bin": "price_binary",
    "mul": "price_multiclass",
}

In [ ]:
BASE_PATH = f"./results/{__DATASET.name.replace(' ', '_')}"
SEED = 202605
K = list(range(2, 11)) + list(range(20, 101, 10))
ALGO_NAME = {
    str(ClassicMondrian): "mondrian",
    str(KMember): "kmember",
    str(Incognito): "incognito",
}

In [ ]:
try:
    _unique_df = pd.read_csv(f"{BASE_PATH}/unique_df.csv")
except:
    _unique_df = create_unique_df(__DATASET, __DATASET.qids, TARGETS["mul"])
    _unique_df.to_csv(f"{BASE_PATH}/unique_df.csv", index=False)
DATASET = SampleDataset(__DATASET, _unique_df)

SPLITS_IDX = []
N_SPLITS = 10

try:
    with open(f"{BASE_PATH}/splits.json", "r") as f:
        SPLITS_IDX = json.load(f)["splits"]
except:
    for run_i, (train_idx, test_idx) in enumerate(
        StratifiedShuffleSplit(
            n_splits=N_SPLITS,
            test_size=0.2,
            random_state=SEED,
        ).split(_unique_df, _unique_df[TARGETS["mul"]])
    ):
        SPLITS_IDX.append(
            {"train_idx": train_idx.tolist(), "test_idx": test_idx.tolist()}
        )

    with open(f"{BASE_PATH}/splits.json", "w") as f:
        json.dump({"splits": SPLITS_IDX}, f, indent=4)

In [ ]:
PIPES = {
    "reg": make_default_ML_pipe("continuous", SEED),
    "bin": make_default_ML_pipe("binary", SEED),
    "mul": make_default_ML_pipe("multiclass", SEED),
}

In [ ]:
def run_exp_org(
    split_indices: list[dict],
    base_dataset: Dataset,
    pipes: dict,
    features: list,
    targets: dict,
):
    """
    Run ML tasks on the original data.
    Metrics to be computed:
    - ML_REG_RMSE: (Regression) Root Mean Squared Error
    - ML_REG_R2: (Regression) R2
    - ML_CLS_BIN_F1: (Classification - Binary) F1 Score
    - ML_CLS_MUL_F1: (Classification - Multi-class) F1 Score
    """

    for i, split in enumerate(split_indices):
        split_path = f"{BASE_PATH}/split_{i}"
        os.makedirs(split_path, exist_ok=True)

        SAMPLE, VALIDATION = get_data(base_dataset, split)

        org_results = BASE_RESULTS.copy()
        org_results["METHOD"] = "original"
        org_results["ML_REG_RMSE"], org_results["ML_REG_R2"] = get_ML_reg_result(
            pipes["reg"], features, targets["reg"], SAMPLE.df, VALIDATION
        )
        org_results["ML_CLS_BIN_F1"] = get_ML_cls_result(
            pipes["bin"], features, targets["bin"], SAMPLE.df, VALIDATION
        )
        org_results["ML_CLS_MUL_F1"] = get_ML_cls_result(
            pipes["mul"], features, targets["mul"], SAMPLE.df, VALIDATION
        )
        pd.DataFrame(org_results, index=[0]).to_csv(
            f"{split_path}/results_org.csv", index=False
        )

In [ ]:
def run_exp_anon(
    algo_class: LocalRecodingAlgorithm,
    all_k: list[int],
    split_indices: list[dict],
    base_dataset: Dataset,
    pipes: dict,
    features: list,
    targets: dict,
    seed: int = None,
):
    """
    Anonymize the data and compute all metrics
    Metrics to be computed:
    - K: k-anonymity
    - RUN_TIME: Run time
    - UT_CAVG: Normalized Average Equivalence Class
    - UT_DM: Discernibility
    - UT_NCP: Normalized Certainty Penalty
    - UT_RM: Regression Penalty
    - UT_CM_BIN: Classification Penalty (Binary)
    - UT_CM_MUL: Classification Penalty (Multi-class)
    - ML_REG_RMSE: (Regression) Root Mean Squared Error
    - ML_REG_R2: (Regression) R2
    - ML_CLS_BIN_F1: (Classification - Binary) F1 Score
    - ML_CLS_MUL_F1: (Classification - Multi-class) F1 Score
    - VUL_DBRL: Vulnerability against Distance-based Record Linkage Attack
    """
    for i, split in enumerate(split_indices):
        print(f"--- Split {i} ---")
        split_path = f"{BASE_PATH}/split_{i}"
        os.makedirs(split_path, exist_ok=True)

        SAMPLE, VALIDATION = get_data(base_dataset, split)

        LAST_GEN_SOLUTION = None
        LAST_RESULTS = BASE_RESULTS.copy()
        LAST_ATTACK_RESULTS = None
        LAST_K = 1

        for k in all_k:
            split_k_path = f"{split_path}/k_{k}"
            os.makedirs(split_k_path, exist_ok=True)

            RESULTS = BASE_RESULTS.copy()
            RESULTS["METHOD"] = ALGO_NAME[str(algo_class)]
            RESULTS["K"] = k

            ALGO, RESULTS["RUN_TIME"] = anonymize_data(algo_class, SAMPLE, k, seed)

            GENERALIZATION = (
                None
                if issubclass(algo_class, LocalRecodingAlgorithm)
                else ALGO.best_generalization_solution
            )

            if (GENERALIZATION != LAST_GEN_SOLUTION) or (GENERALIZATION is None):
                LAST_RESULTS = get_ut_and_ml_metrics(
                    ALGO, VALIDATION, pipes, features, targets, GENERALIZATION
                )
                LAST_ATTACK_RESULTS, LAST_RESULTS["VUL_DBRL"] = get_attack_results(
                    ALGO, SAMPLE, SAMPLE.qids + [targets["mul"]], GENERALIZATION, seed
                )
                if GENERALIZATION is not None:
                    LAST_GEN_SOLUTION = GENERALIZATION.copy()
            else:
                LAST_RESULTS["UT_CAVG"] = LAST_RESULTS["UT_CAVG"] * LAST_K / k
            LAST_K = k

            split_k_anon_path = f"{split_k_path}/{ALGO_NAME[str(algo_class)]}"
            os.makedirs(split_k_anon_path, exist_ok=True)

            ALGO.anon_data.to_csv(f"{split_k_anon_path}/anon_data.csv", index=False)
            RESULTS.update(LAST_RESULTS)
            pd.DataFrame(RESULTS, index=[0]).to_csv(
                f"{split_k_anon_path}/results.csv", index=False
            )

            LAST_ATTACK_RESULTS.to_csv(f"{split_k_anon_path}/attack.csv", index=False)

            print(f"Results for k={k} saved!")

        print("---------------")

In [ ]:
run_exp_org(
    SPLITS_IDX[:1],
    DATASET,
    PIPES,
    DATASET.qids,
    TARGETS,
)

In [ ]:
run_exp_anon(
    ClassicMondrian,
    K,
    SPLITS_IDX,
    DATASET,
    PIPES,
    DATASET.qids,
    TARGETS,
    SEED,
)

In [ ]:
run_exp_anon(
    KMember,
    K,
    SPLITS_IDX,
    DATASET,
    PIPES,
    DATASET.qids,
    TARGETS,
    SEED,
)

In [ ]:
run_exp_anon(
    Incognito,
    K,
    SPLITS_IDX,
    DATASET,
    PIPES,
    DATASET.qids,
    TARGETS,
    SEED,
)

In [ ]:
all_results = []
for i in range(len(SPLITS_IDX)):
    for k in K:
        for algo in ["mondrian", "kmember", "incognito"]:
            r = pd.read_csv(f"{BASE_PATH}/split_{i}/k_{k}/{algo}/results.csv")
            all_results.append(r)
    r = pd.read_csv(f"{BASE_PATH}/split_{i}/results_org.csv")
    all_results.append(r)

all_results_df = pd.concat(all_results)
all_results_df.K = all_results_df.K.astype("int")
all_results_df.groupby(["METHOD", "K"], as_index=False).mean().to_csv(
    f"{BASE_PATH}/all_results.csv", index=False
)

In [ ]:
all_results_df